## Persistent Landing

**Importing Useful Libraries**

In [1]:
import os
import time
import boto3
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
# Create some sub-buckets inside persistent-landing, one per format
s3.put_object(Bucket="landing-zone", Key="persistent-landing/structured/raw/") # Sub-bucket CSV
s3.put_object(Bucket="landing-zone", Key="persistent-landing/unstructured/image/") # Sub-bucket IMAGE
s3.put_object(Bucket="landing-zone", Key="persistent-landing/semistructured/")

{'ResponseMetadata': {'RequestId': '18B27CCC2EE390D4',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18B27CCC2EE390D4',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '9020',
   'x-ratelimit-remaining': '9020',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 24 May 2026 11:26:59 GMT'},
  'RetryAttempts': 0},
 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"'}

In [4]:
# This function checks an object's ContentType (via head_object) and classifies it as "image" or "csv"
# based on whether the type starts with image/ or other.
def classify_object_by_head(client, bucket, key):
    # ask S3 for ContentType
    head = client.head_object(Bucket=bucket, Key=key)
    ct = head.get("ContentType", "")
    if ct.startswith("image/"):
        return "image"
    else:
        return "structured"

In [5]:
# This function moves all files from the source_prefix folder to the dest_prefix folder,
# classifying each file as image or csv (based on ContentType), renaming it with a timestamped filename (ingestion time),
# copying it to the appropriate subfolder (image/, csv/), and then deleting the original files in the source_prefix.
def move_files(client, bucket, source_prefix="temporal-landing/", dest_prefix="persistent-landing/"):
    
    paginator = client.get_paginator("list_objects_v2") # It returns objects in pages and not all at once.

    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            
            src_key = obj["Key"]

            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # classify
            category = classify_object_by_head(client, bucket, src_key)
            # get file extension
            ext = src_key.split('.')[-1].split('?')[0]
            # new filename = timestamp + original extension
            ts = int(time.time() * 1000)  # milliseconds

            if category == "structured":
                category = "structured/"+"raw"
                new_filename = f"{os.path.splitext(os.path.basename(src_key))[0]}_{ts}.{ext}"
            else:
                new_filename = f"{category}_{ts}.{ext}"

            # build destination key
            if category == "image":
                category = "unstructured/"+category

            dest_key = f"{dest_prefix}{category}/{new_filename}"

            # copy then delete
            client.copy_object(Bucket=bucket, CopySource={"Bucket": bucket, "Key": src_key}, Key=dest_key)
            client.delete_object(Bucket=bucket, Key=src_key)

            print(f"Moved: {src_key} -> {dest_key}")

In [6]:
# Moving files from Temporal Landing to Persistent Landing and removing temporal files
move_files(s3, "landing-zone", "temporal-landing/", "persistent-landing/")

Moved: temporal-landing/co2-emission-by-vehicles.csv -> persistent-landing/structured/raw/co2-emission-by-vehicles_1779622021500.csv
Moved: temporal-landing/global_warming_dataset.csv -> persistent-landing/structured/raw/global_warming_dataset_1779622021583.csv
Moved: temporal-landing/image_0_0.jpg -> persistent-landing/unstructured/image/image_1779622021970.jpg
Moved: temporal-landing/image_0_1.jpg -> persistent-landing/unstructured/image/image_1779622022028.jpg
Moved: temporal-landing/image_0_10.jpg -> persistent-landing/unstructured/image/image_1779622022092.jpg
Moved: temporal-landing/image_0_11.jpg -> persistent-landing/unstructured/image/image_1779622022147.jpg
Moved: temporal-landing/image_0_12.jpg -> persistent-landing/unstructured/image/image_1779622022201.jpg
Moved: temporal-landing/image_0_13.jpg -> persistent-landing/unstructured/image/image_1779622022257.jpg
Moved: temporal-landing/image_0_14.jpg -> persistent-landing/unstructured/image/image_1779622022309.jpg
Moved: tempo